In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=41d38d0a09c96d6901c694f0dc822bd6d241bf7ccc624a3f5ed7bb2f902355d1
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [6]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol with an attacker,
# to demonstrate that the attacker can be detected.

# ── Eve's strategy (intercept-resend attack) ─────────────────────────────────
# Eve intercepts every qubit Alice sends.
# She picks a random measurement basis for each qubit and measures it.
# She then re-encodes her measured result in the same basis she used,
# and forwards the new qubit to Bob.
#
# Why this introduces errors:
#   - Eve guesses the right basis ~50% of the time.
#   - When she guesses wrong, she collapses the qubit to the wrong state.
#   - Bob then has a 50% chance of getting the wrong bit at those positions.
#   - Net effect: ~25% error rate in the sifted key.
#   - Alice and Bob detect this because their sifted keys no longer match.

backend = BasicSimulator()

In [7]:
# ── Quantum random number generation ────────────────────────────────────────
# Measure n qubits each prepared in |+⟩ = H|0⟩ = 1/√2 (|0⟩ + |1⟩).
# Each measurement yields 0 or 1 with equal probability — no classical RNG used.

def quantum_random_bits(n):
    """Generate n random bits by measuring n qubits each prepared in |+⟩."""
    qc = QuantumCircuit(n, n)
    qc.h(range(n))            # H|0⟩ = |+⟩ for every qubit
    qc.measure(range(n), range(n))
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    # Qiskit returns bits big-endian; reverse so index 0 = qubit 0
    return [int(b) for b in reversed(bitstring)]

# ── Alice: encoding ──────────────────────────────────────────────────────────
#   basis 0 (Z):  bit 0 → |0⟩,  bit 1 → |1⟩
#   basis 1 (X):  bit 0 → |+⟩,  bit 1 → |−⟩

def alice_encode(bit, basis):
    """[ALICE] Encode `bit` in `basis` (0=Z, 1=X). Returns a 1-qubit circuit."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

# ── Shared measurement helper ─────────────────────────────────────────────────
# Used by both Eve and Bob to measure a qubit circuit in a chosen basis.

def measure_qubit(qc, basis):
    """Measure qubit in `qc` using `basis` (0=Z, 1=X). Returns 0 or 1."""
    if basis == 1:
        qc.h(0)       # rotate from X basis to Z before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])

# ── Eve: re-encode after measuring ───────────────────────────────────────────
# Eve creates a fresh qubit encoding her measured result in her chosen basis,
# and sends it on to Bob. This is the key step that disturbs the state.

def eve_reencode(measured_bit, basis):
    """[EVE] Re-encode `measured_bit` in `basis` to send to Bob."""
    qc = QuantumCircuit(1, 1)
    if measured_bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

In [8]:
# ── Run the BB84 protocol with Eve intercepting all qubits ───────────────────

n = 20              # number of qubits Alice sends
THRESHOLD = 0.10    # error rate above this → attack detected

print("=" * 60)
print("BB84 Protocol — With Attacker (Eve)")
print("=" * 60)

# ── Step 1: Alice generates random bits and bases ────────────────────────────
alice_bits  = quantum_random_bits(n)
alice_bases = quantum_random_bits(n)   # 0 = Z basis, 1 = X basis
print(f"\n[ALICE] Random bits:  {alice_bits}")
print(f"[ALICE] Random bases: {alice_bases}  (0=Z, 1=X)")

# ── Step 2: Eve intercepts every qubit ──────────────────────────────────────
# Eve picks a random basis, measures, then re-encodes and forwards to Bob.
# Alice and Bob are unaware of this interception.

eve_bases        = quantum_random_bits(n)
eve_results      = []
forwarded_qubits = []   # circuits Eve sends to Bob

print(f"\n[EVE]   Random bases: {eve_bases}  (0=Z, 1=X)  ← secret")

for i in range(n):
    qc_alice  = alice_encode(alice_bits[i], alice_bases[i])  # Alice's original qubit
    eve_bit   = measure_qubit(qc_alice, eve_bases[i])        # [EVE] measures it
    eve_results.append(eve_bit)
    qc_resent = eve_reencode(eve_bit, eve_bases[i])          # [EVE] re-encodes and forwards
    forwarded_qubits.append(qc_resent)

print(f"[EVE]   Measured:     {eve_results}  ← secret")

# ── Step 3: Bob measures Eve's forwarded qubits ──────────────────────────────
bob_bases   = quantum_random_bits(n)
bob_results = []

for i in range(n):
    bit = measure_qubit(forwarded_qubits[i], bob_bases[i])   # [BOB] measures
    bob_results.append(bit)

print(f"\n[BOB]   Random bases: {bob_bases}  (0=Z, 1=X)")
print(f"[BOB]   Measurements: {bob_results}")

# ── Step 4: Sifting — Alice and Bob compare bases publicly ───────────────────
sifted_alice = []
sifted_bob   = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"\n[PUBLIC] Bases compared. {len(sifted_alice)}/{n} positions matched.")
print(f"\n[ALICE] Sifted key: {sifted_alice}")
print(f"[BOB]   Sifted key: {sifted_bob}")

# ── Step 5: Error checking — detect the attacker ─────────────────────────────
# Alice and Bob compare a subset of sifted key bits over a public channel.
# Any mismatch reveals that the channel was disturbed (i.e. Eve was present).
# In a real deployment they would sacrifice roughly half the sifted bits for
# this check and keep the rest as the secret key.

errors     = sum(a != b for a, b in zip(sifted_alice, sifted_bob))
error_rate = errors / len(sifted_alice) if sifted_alice else 0

print(f"\n[CHECK] Errors:     {errors}/{len(sifted_alice)}")
print(f"[CHECK] Error rate: {error_rate:.1%}  (threshold = {THRESHOLD:.0%})")

if error_rate > THRESHOLD:
    print("[CHECK]  ATTACK DETECTED — error rate exceeds threshold!")
    print("         Alice and Bob abort the key exchange.")
else:
    print("[CHECK]  No attack detected (error rate within threshold).")

print(f"\n── Why ~25% errors? ────────────────────────────────────────")
print(f"  Eve guesses the correct basis 50% of the time.")
print(f"  When she guesses wrong, Bob gets a random bit → 50% error.")
print(f"  Overall: 50% × 50% = ~25% error rate in the sifted key.")

BB84 Protocol — With Attacker (Eve)

[ALICE] Random bits:  [1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0]
[ALICE] Random bases: [1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0]  (0=Z, 1=X)

[EVE]   Random bases: [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0]  (0=Z, 1=X)  ← secret
[EVE]   Measured:     [1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0]  ← secret

[BOB]   Random bases: [0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0]  (0=Z, 1=X)
[BOB]   Measurements: [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0]

[PUBLIC] Bases compared. 8/20 positions matched.

[ALICE] Sifted key: [1, 0, 1, 1, 0, 1, 0, 0]
[BOB]   Sifted key: [0, 1, 0, 1, 1, 1, 0, 0]

[CHECK] Errors:     4/8
[CHECK] Error rate: 50.0%  (threshold = 10%)
[CHECK]  ATTACK DETECTED — error rate exceeds threshold!
         Alice and Bob abort the key exchange.

── Why ~25% errors? ────────────────────────────────────────
  Eve guesses the correct basi